In [1]:
import torch

In [2]:
class MLP(torch.nn.Module):

    def __init__(self, num_features, num_classes):

        super().__init__()

        self.num_features = num_features
        self.classes = num_classes

        self.layers = torch.nn.Sequential(
            torch.nn.Linear(self.num_features, 50),
            torch.nn.ReLU(),
            torch.nn.Linear(50, 25),
            torch.nn.ReLU(),
            torch.nn.Linear(25, self.classes)
        )

    def forward(self, X):

        logits = self.layers(X).flatten()

        return logits

In [3]:
from torchvision import datasets

In [4]:
from torchvision.transforms import ToTensor

In [5]:
training_data = datasets.FashionMNIST("../datasets/fashionmnist/", 
                                      train=True, 
                                      transform=ToTensor(), 
                                      download=True,
                                      
                                     )
test_data = datasets.FashionMNIST("../datasets/fashionmnist/", 
                                      train=False, 
                                      transform=ToTensor(), 
                                      download=True
                                     )

In [6]:
from torch.utils.data import DataLoader, Dataset

In [7]:
class TestDataset(Dataset):

    def __init__(self, predictors, target):

        super().__init__()
        self.predictors = torch.tensor(predictors)
        self.target = torch.tensor(target)

    def __getitem__(self, index):

        return self.predictors[index], self.target[index]

    def __len__(self):

        return self.target.shape[0]
        

    

In [8]:
from sklearn import datasets

In [9]:
df = datasets.load_breast_cancer()

In [10]:
X = df['data']
y=df['target']

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [13]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train)

In [14]:
from sklearn.preprocessing import StandardScaler

In [15]:
sd_scaler = StandardScaler()

In [16]:
sd_scaler.fit(X_train)

StandardScaler()

In [17]:
X_train_scaled = sd_scaler.transform(X_train)
X_test_scaled = sd_scaler.transform(X_test)
X_val_scaled = sd_scaler.transform(X_val)

In [18]:
X_train_scaled[1,:]

array([-0.89634955, -0.85413705, -0.86412881, -0.81128793,  0.2581729 ,
        0.01375623, -0.48639237, -0.85242923,  0.15025064,  0.73113589,
       -0.89778499, -1.29726421, -0.75012712, -0.63515852, -0.89566534,
        0.2833147 , -0.15371884, -0.83993808, -1.04132989,  0.29397337,
       -0.88799218, -1.02575007, -0.77794236, -0.77834845, -0.00234715,
        0.83253779,  0.27501136, -0.53693045, -0.5242495 ,  1.23723626])

In [19]:
y_train[1]

np.int64(1)

In [20]:
X_train_scaled.dtype

dtype('float64')

In [21]:
y_train.dtype

dtype('int64')

In [22]:
train_dataset = TestDataset(X_train_scaled.astype('float32'), y_train.astype('float32'))
test_dataset = TestDataset(X_test_scaled.astype('float32'), y_test.astype('float32'))
val_dataset = TestDataset(X_val_scaled.astype('float32'), y_val.astype('float32'))

In [23]:
batch_size = 8

In [24]:
train_dl = DataLoader(train_dataset, batch_size=batch_size, shuffle=True )
test_dl = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
val_dl = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [25]:
model = MLP(30, 1)

In [26]:
optim = torch.optim.SGD(model.parameters(), lr=0.001)

In [27]:
epochs = 10

In [28]:
import torch.nn.functional as F

In [29]:
loss_list = []

for epoch in range(epochs):

    model.train()

    for batch_idx, (features, targets) in enumerate(train_dl):

        outputs = model(features)

        loss = F.binary_cross_entropy_with_logits(outputs, targets)

        optim.zero_grad()
        loss.backward()
        optim.step()

        loss_list.append(loss.item())        

In [31]:
model.eval()

MLP(
  (layers): Sequential(
    (0): Linear(in_features=30, out_features=50, bias=True)
    (1): ReLU()
    (2): Linear(in_features=50, out_features=25, bias=True)
    (3): ReLU()
    (4): Linear(in_features=25, out_features=1, bias=True)
  )
)

In [33]:
y_pred_list=[]
y_true_list=[]

with torch.no_grad():
    
    for X, y  in test_dl:
        y_pred = model(X)
        y_pred_list.append(y_pred)
        y_true_list.append(y)
        

In [34]:
y_pred_list

[tensor([ 0.2860, -0.1107,  0.2055,  0.3854,  0.1655,  0.0744,  0.3102,  0.2801]),
 tensor([ 0.4023,  0.2903,  0.2723, -0.0690,  0.4017,  0.3291,  0.2279,  0.3473]),
 tensor([ 0.3004,  0.1312,  0.3556,  0.2749, -0.2086,  0.2593,  0.2292,  0.1265]),
 tensor([ 0.3050,  0.1924, -0.0839,  0.0880,  0.3014,  0.4179,  0.0602,  0.0285]),
 tensor([ 0.2268,  0.0185,  0.1804,  0.2996, -0.1812,  0.1883,  0.3893,  0.4009]),
 tensor([-0.1662,  0.3327,  0.2728,  0.0900,  0.4164,  0.1868,  0.3274,  0.4352]),
 tensor([-0.5082,  0.3618,  0.3279,  0.0991,  0.2509,  0.3101,  0.1871,  0.2294]),
 tensor([0.2877, 0.0688, 0.3222, 0.3792, 0.2958, 0.2994, 0.3691, 0.4243]),
 tensor([ 0.3797, -0.1796,  0.3768,  0.3297,  0.2577,  0.1993,  0.3137,  0.3664]),
 tensor([0.1261, 0.3205, 0.3398, 0.3103, 0.3127, 0.2836, 0.3082, 0.3688]),
 tensor([0.3336, 0.3461, 0.2651, 0.3254, 0.1118, 0.2177, 0.1802, 0.1626]),
 tensor([ 0.2683,  0.3793,  0.3004, -0.3260,  0.2593,  0.2381,  0.1373,  0.2750]),
 tensor([0.4475, 0.2776, 0.3

In [35]:
y_true_list

[tensor([0., 0., 0., 1., 0., 0., 1., 1.]),
 tensor([1., 1., 1., 0., 1., 1., 1., 1.]),
 tensor([1., 0., 1., 1., 0., 1., 1., 0.]),
 tensor([1., 0., 0., 0., 1., 1., 0., 0.]),
 tensor([0., 0., 0., 1., 0., 0., 1., 1.]),
 tensor([0., 1., 1., 0., 1., 0., 1., 1.]),
 tensor([0., 1., 1., 0., 1., 1., 0., 1.]),
 tensor([1., 0., 1., 1., 1., 1., 1., 1.]),
 tensor([1., 0., 1., 1., 0., 0., 1., 1.]),
 tensor([0., 1., 1., 1., 1., 1., 1., 1.]),
 tensor([1., 1., 1., 1., 0., 0., 0., 0.]),
 tensor([1., 1., 1., 0., 1., 0., 0., 1.]),
 tensor([1., 1., 1., 0., 0., 1., 1., 0.]),
 tensor([1., 1., 1., 1., 1., 1., 0., 1.]),
 tensor([0., 1.])]